This project focuses on demonstrating practical NumPy skills including:

Vectorized computation,
Boolean masking,
Broadcasting,
Statistical analysis and
Matrix operations

The goal is to analyze  risk patterns associated with loan default using pure NumPy operations wherever possible.

In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv("Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [4]:
# Numerical features
numerical_columns = [
    "Age", "Income", "LoanAmount", "CreditScore",
    "MonthsEmployed", "NumCreditLines",
    "InterestRate", "LoanTerm", "DTIRatio"
]

X = df[numerical_columns].to_numpy()

y = df["Default"].to_numpy()

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (255347, 9)
Shape of y: (255347,)


Inspection of the  NumPy arrays structure before performing analysis.

In [5]:
print("First 5 rows of X:\n", X[:5])
print("\nFirst 5 values of y:\n", y[:5])

print("\nData type of X:", X.dtype)
print("Data type of y:", y.dtype)

print("\nNumber of samples:", X.shape[0])
print("Number of features:", X.shape[1])

First 5 rows of X:
 [[5.60000e+01 8.59940e+04 5.05870e+04 5.20000e+02 8.00000e+01 4.00000e+00
  1.52300e+01 3.60000e+01 4.40000e-01]
 [6.90000e+01 5.04320e+04 1.24440e+05 4.58000e+02 1.50000e+01 1.00000e+00
  4.81000e+00 6.00000e+01 6.80000e-01]
 [4.60000e+01 8.42080e+04 1.29188e+05 4.51000e+02 2.60000e+01 3.00000e+00
  2.11700e+01 2.40000e+01 3.10000e-01]
 [3.20000e+01 3.17130e+04 4.47990e+04 7.43000e+02 0.00000e+00 3.00000e+00
  7.07000e+00 2.40000e+01 2.30000e-01]
 [6.00000e+01 2.04370e+04 9.13900e+03 6.33000e+02 8.00000e+00 4.00000e+00
  6.51000e+00 4.80000e+01 7.30000e-01]]

First 5 values of y:
 [0 0 1 0 0]

Data type of X: float64
Data type of y: int64

Number of samples: 255347
Number of features: 9


Default Distribution Using Boolean Masking

In [6]:
# Boolean masks
default_mask = y == 1
non_default_mask = y == 0

num_default = np.sum(default_mask)
num_non_default = np.sum(non_default_mask)

print("Number of Defaulters:", num_default)
print("Number of Non-Defaulters:", num_non_default)

print("Default Rate:", num_default / len(y))

Number of Defaulters: 29653
Number of Non-Defaulters: 225694
Default Rate: 0.11612824901017048


Comparing Financial Features Between Groups

In [7]:
feature_names = numerical_columns

mean_default = np.mean(X[default_mask], axis=0)
mean_non_default = np.mean(X[non_default_mask], axis=0)

for i, feature in enumerate(feature_names):
    print(f"{feature}:")
    print(f"   Mean (Default=1): {mean_default[i]:.2f}")
    print(f"   Mean (Default=0): {mean_non_default[i]:.2f}")
    print()

Age:
   Mean (Default=1): 36.56
   Mean (Default=0): 44.41

Income:
   Mean (Default=1): 71844.72
   Mean (Default=0): 83899.17

LoanAmount:
   Mean (Default=1): 144515.31
   Mean (Default=0): 125353.66

CreditScore:
   Mean (Default=1): 559.29
   Mean (Default=0): 576.23

MonthsEmployed:
   Mean (Default=1): 50.24
   Mean (Default=0): 60.76

NumCreditLines:
   Mean (Default=1): 2.59
   Mean (Default=0): 2.49

InterestRate:
   Mean (Default=1): 15.90
   Mean (Default=0): 13.18

LoanTerm:
   Mean (Default=1): 36.05
   Mean (Default=0): 36.02

DTIRatio:
   Mean (Default=1): 0.51
   Mean (Default=0): 0.50



Correlation Analysis with Target Variable

In [8]:
# Combine X and y temporarily
data_matrix = np.column_stack((X, y))

# Compute correlation matrix
corr_matrix = np.corrcoef(data_matrix.T)

# Last row corresponds to correlation with y
correlation_with_default = corr_matrix[-1, :-1]

for feature, corr in zip(numerical_columns, correlation_with_default):
    print(f"{feature}: {corr:.4f}")

Age: -0.1678
Income: -0.0991
LoanAmount: 0.0867
CreditScore: -0.0342
MonthsEmployed: -0.0974
NumCreditLines: 0.0283
InterestRate: 0.1313
LoanTerm: 0.0005
DTIRatio: 0.0192


Feature Index Mapping

In [10]:
feature_to_index = {name: i for i, name in enumerate(numerical_columns)}

print(feature_to_index)

{'Age': 0, 'Income': 1, 'LoanAmount': 2, 'CreditScore': 3, 'MonthsEmployed': 4, 'NumCreditLines': 5, 'InterestRate': 6, 'LoanTerm': 7, 'DTIRatio': 8}


Boolean Masking (Risk Segmentation by Income)

In [11]:
income_col = X[:, feature_to_index["Income"]]

high_income_mask = income_col > 100000
low_income_mask = income_col < 50000

print("Default Rate (High Income):", y[high_income_mask].mean())
print("Default Rate (Low Income):", y[low_income_mask].mean())

Default Rate (High Income): 0.09103791043197801
Default Rate (Low Income): 0.17162914548535463


Percentile-Based Credit Risk Segmentation

In [12]:
credit_col = X[:, feature_to_index["CreditScore"]]

q25 = np.percentile(credit_col, 25)
q75 = np.percentile(credit_col, 75)

low_credit_mask = credit_col <= q25
high_credit_mask = credit_col >= q75

print("Default Rate (Lowest 25% Credit Scores):", y[low_credit_mask].mean())
print("Default Rate (Highest 25% Credit Scores):", y[high_credit_mask].mean())

Default Rate (Lowest 25% Credit Scores): 0.13055815403804177
Default Rate (Highest 25% Credit Scores): 0.10183187970864985


Rank Features by Correlation Strength


In [13]:
abs_corr = np.abs(correlation_with_default)

sorted_indices = np.argsort(abs_corr)[::-1]

print("Feature Ranking by Correlation with Default:\n")

for idx in sorted_indices:
    print(f"{numerical_columns[idx]}: {correlation_with_default[idx]:.4f}")

Feature Ranking by Correlation with Default:

Age: -0.1678
InterestRate: 0.1313
Income: -0.0991
MonthsEmployed: -0.0974
LoanAmount: 0.0867
CreditScore: -0.0342
NumCreditLines: 0.0283
DTIRatio: 0.0192
LoanTerm: 0.0005


Standardization + Covariance Matrix

In [14]:
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)

cov_matrix = np.cov(X_scaled.T)

print("Covariance Matrix Shape:", cov_matrix.shape)

Covariance Matrix Shape: (9, 9)


Vectorization vs Loop

In [15]:
import time

# Loop version
start = time.time()
mean_loop = np.zeros(X.shape[1])
for i in range(X.shape[1]):
    mean_loop[i] = np.mean(X[:, i])
loop_time = time.time() - start

# Vectorized version
start = time.time()
mean_vectorized = X.mean(axis=0)
vectorized_time = time.time() - start

print("Loop Time:", loop_time)
print("Vectorized Time:", vectorized_time)

Loop Time: 0.003315448760986328
Vectorized Time: 0.002597808837890625
